# Student Zero-Shot and LoRA SFT Experiments

Use this notebook to run the next project stage: validate teacher data, evaluate the base student zero-shot, build SFT files, train one or more LoRA configs, and compare output metrics.

Recommended flow: run zero-shot first, inspect metrics and outputs, then enable one small SFT config before running a larger config.

## 1. Colab Pull / Local Setup

Run this first. In Colab it clones or pulls the GitHub repo and switches into the project directory. Locally it leaves your current checkout alone.

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kdnehihi/strategy-distill-rl.git"
REPO_DIR = Path("/content/strategy-distill-rl")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    if REPO_DIR.exists():
        print(f"Pulling latest repo in {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    else:
        print(f"Cloning repo to {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"Local run detected. Current directory: {Path.cwd()}")

print(f"Working directory: {Path.cwd()}")


## 2. Install Dependencies

Set `INSTALL_DEPENDENCIES = True` on a fresh runtime. Keep it false if your environment is already ready.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Skipping dependency install. Set INSTALL_DEPENDENCIES=True if needed.")


## 3. Experiment Config

Change the flags and config list here. Keep the first run small; then increase samples or epochs once the output format looks stable.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"
TEACHER_PATH = "data/gsm8k_teacher_preview_5000.jsonl"
EVAL_INPUT_PATH = "data/gsm8k_clean_test.jsonl"
SFT_TRAIN_PATH = "data/sft_strategy_train.jsonl"
SFT_VAL_PATH = "data/sft_strategy_val.jsonl"

# From the Drive screenshot: My Drive > RL > Data
DRIVE_DATA_DIR = "/content/drive/MyDrive/RL/Data"

RUN_VALIDATE_TEACHER = True
RUN_ZERO_SHOT = False
RUN_BUILD_SFT = True
RUN_TRAINING = True
RUN_ADAPTER_EVAL = True

ZERO_SHOT_NUM_SAMPLES = 100
FINAL_EVAL_NUM_SAMPLES = 100
EVAL_BATCH_SIZE = 8
EVAL_MAX_NEW_TOKENS = 256

# Default SFT mode trains only the smoke config. Add more names after checking outputs.
ACTIVE_TRAIN_CONFIG_NAMES = ["smoke_r8_a16_300"]

TRAIN_CONFIGS = [
    {
        "name": "smoke_r8_a16_300",
        "max_train_samples": 300,
        "max_val_samples": 100,
        "epochs": 1.0,
        "lr": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
    {
        "name": "balanced_r16_a32_4000",
        "max_train_samples": 4000,
        "max_val_samples": 298,
        "epochs": 1.0,
        "lr": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
    {
        "name": "stronger_r32_a64_4000",
        "max_train_samples": 4000,
        "max_val_samples": 298,
        "epochs": 2.0,
        "lr": 1e-4,
        "lora_r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
]

RUNS_DIR = Path("runs/student_sft")
CHECKPOINTS_DIR = Path("checkpoints/student_sft")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


## 4. Helpers

These helpers run repo scripts and load metric JSON files into a comparison table.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd


def run_command(args):
    print("$", " ".join(str(arg) for arg in args))
    subprocess.run([str(arg) for arg in args], check=True)


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path, limit=None):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if limit is not None and len(rows) >= limit:
                break
            rows.append(json.loads(line))
    return rows


def metric_row(name, metrics_path, train_metrics_path=None):
    metrics = load_json(metrics_path)
    row = {
        "run": name,
        "total": metrics.get("total"),
        "accuracy": metrics.get("accuracy"),
        "loose_math_accuracy": metrics.get("loose_math_accuracy"),
        "format_valid_rate": metrics.get("format_valid_rate"),
        "usable_rate": metrics.get("usable_rate"),
        "correct": metrics.get("correct"),
        "loose_correct": metrics.get("loose_correct"),
        "format_valid": metrics.get("format_valid"),
        "usable": metrics.get("usable"),
        "metrics_path": str(metrics_path),
    }
    if train_metrics_path and Path(train_metrics_path).exists():
        train_metrics = load_json(train_metrics_path)
        row["eval_loss"] = train_metrics.get("eval", {}).get("eval_loss")
        row["train_loss"] = train_metrics.get("train", {}).get("train_loss")
    return row


## 5. Ensure Data Files

`data/*.jsonl` files are intentionally gitignored, so a fresh Colab pull will not include them. This cell mounts Google Drive, copies files from `My Drive/RL/Data`, and only regenerates GSM8K if the clean files are still missing.

In [ ]:
import shutil

Path("data").mkdir(parents=True, exist_ok=True)


def mount_drive_if_needed():
    if not IN_COLAB:
        return

    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        print("Google Drive already mounted.")
        return

    print("Mounting Google Drive...")
    from google.colab import drive

    drive.mount("/content/drive")


def copy_if_exists(source_path, target_path):
    source = Path(source_path)
    target = Path(target_path)
    if not source.exists():
        return False

    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        print(f"Already exists: {target}")
    else:
        shutil.copy2(source, target)
        print(f"Copied {source} -> {target}")
    return True


def copy_drive_data_files():
    mount_drive_if_needed()

    drive_dir = Path(DRIVE_DATA_DIR)
    if not drive_dir.exists():
        print(f"Drive data directory not found: {drive_dir}")
        return

    print(f"Using Drive data directory: {drive_dir}")
    expected_files = [
        "gsm8k_clean_train.jsonl",
        "gsm8k_clean_test.jsonl",
        "gsm8k_teacher_preview_5000.jsonl",
        "sft_strategy_train.jsonl",
        "sft_strategy_val.jsonl",
    ]
    for filename in expected_files:
        copy_if_exists(drive_dir / filename, Path("data") / filename)


def ensure_clean_gsm8k_files():
    required = [Path(EVAL_INPUT_PATH), Path("data/gsm8k_clean_train.jsonl")]
    if all(path.exists() for path in required):
        print("Clean GSM8K files already exist.")
        return

    print("Missing clean GSM8K files after Drive copy. Running scripts/prepare_gsm8k.py first...")
    run_command(["python", "-B", "scripts/prepare_gsm8k.py"])


def find_teacher_candidate(target_path):
    target = Path(target_path)
    drive_dir = Path(DRIVE_DATA_DIR)
    candidates = [
        target,
        Path.cwd() / target.name,
        Path("/content") / target.name,
        drive_dir / target.name,
        Path("/content/drive/MyDrive") / target.name,
        Path("/content/drive/MyDrive/strategy-distill-rl") / target.name,
        Path("/content/drive/MyDrive/RL/Data") / target.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def ensure_teacher_file():
    target = Path(TEACHER_PATH)
    if target.exists():
        print(f"Teacher file already exists: {target}")
        return

    candidate = find_teacher_candidate(target)
    if candidate is not None:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidate, target)
        print(f"Copied teacher file from {candidate} -> {target}")
        return

    if IN_COLAB:
        print(f"Missing {target}. Please upload your validated teacher JSONL now.")
        print("Expected file name is usually gsm8k_teacher_preview_5000.jsonl.")
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError(f"No file uploaded for {target}")

        uploaded_name = next(iter(uploaded))
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("wb") as f:
            f.write(uploaded[uploaded_name])
        print(f"Uploaded {uploaded_name} -> {target}")
        return

    raise FileNotFoundError(
        f"Missing {target}. Copy your validated teacher JSONL to {target}, "
        "or set TEACHER_PATH to its real local path."
    )


copy_drive_data_files()
ensure_clean_gsm8k_files()
if RUN_VALIDATE_TEACHER or RUN_BUILD_SFT or RUN_TRAINING:
    ensure_teacher_file()


## 6. Validate Teacher Data and Build SFT Files

The teacher file should already contain only usable records. Validation is still the gate before student training.

In [ ]:
if RUN_VALIDATE_TEACHER:
    run_command([
        "python", "-B", "scripts/validate_teacher_dataset.py",
        "--path", TEACHER_PATH,
    ])

if RUN_BUILD_SFT:
    run_command([
        "python", "-B", "scripts/build_sft_dataset.py",
        "--teacher-path", TEACHER_PATH,
        "--train-output", SFT_TRAIN_PATH,
        "--val-output", SFT_VAL_PATH,
        "--train-size", "4000",
    ])


## 7. Zero-Shot Baseline

Strict `accuracy` requires a valid `<final>` block. `loose_math_accuracy` is diagnostic only: it extracts a best-effort numeric answer from raw zero-shot text so we can tell math ability apart from format following. `usable_rate` remains the main pipeline metric.

In [ ]:
baseline_output = RUNS_DIR / "zero_shot_outputs.jsonl"
baseline_metrics = RUNS_DIR / "zero_shot_metrics.json"

if RUN_ZERO_SHOT:
    run_command([
        "python", "-B", "scripts/evaluate_student.py",
        "--model-name", MODEL_NAME,
        "--input-path", EVAL_INPUT_PATH,
        "--output-path", baseline_output,
        "--metrics-path", baseline_metrics,
        "--num-samples", str(ZERO_SHOT_NUM_SAMPLES),
        "--batch-size", str(EVAL_BATCH_SIZE),
        "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
    ])

if baseline_metrics.exists():
    display(pd.DataFrame([metric_row("zero_shot", baseline_metrics)]))

## 8. Inspect Zero-Shot Outputs

Look at a few failed or invalid outputs before training. If zero-shot already follows the format well, keep SFT lighter.

In [ ]:
if baseline_output.exists():
    rows = read_jsonl(baseline_output)
    failed = [row for row in rows if not row.get("is_usable")]
    print(f"total={len(rows)} failed_or_unusable={len(failed)}")
    for row in failed[:3]:
        print("=" * 80)
        print("id:", row["id"])
        print("question:", row["question"])
        print("ground_truth:", row["ground_truth"])
        print("model_answer:", row["model_answer"])
        print("loose_model_answer:", row.get("loose_model_answer"))
        print("is_correct:", row["is_correct"], "is_loose_correct:", row.get("is_loose_correct"), "is_format_valid:", row["is_format_valid"])
        print("format_checks:", row["format_checks"])
        print("raw_model_output:")
        print((row.get("raw_model_output") or "")[:1600])


## 9. Train LoRA SFT Configs

Set `RUN_TRAINING = True` in the config cell when ready. Start with `smoke_r8_a16_300`; only run the larger configs after the smoke run produces sane outputs.

In [ ]:
trained_configs = []

if RUN_TRAINING:
    for cfg in TRAIN_CONFIGS:
        if cfg["name"] not in ACTIVE_TRAIN_CONFIG_NAMES:
            continue
        output_dir = CHECKPOINTS_DIR / cfg["name"]
        run_command([
            "python", "-B", "scripts/train_sft_lora.py",
            "--model-name", MODEL_NAME,
            "--train-path", SFT_TRAIN_PATH,
            "--val-path", SFT_VAL_PATH,
            "--output-dir", output_dir,
            "--max-train-samples", str(cfg["max_train_samples"]),
            "--max-val-samples", str(cfg["max_val_samples"]),
            "--num-train-epochs", str(cfg["epochs"]),
            "--learning-rate", str(cfg["lr"]),
            "--lora-r", str(cfg["lora_r"]),
            "--lora-alpha", str(cfg["lora_alpha"]),
            "--lora-dropout", str(cfg["lora_dropout"]),
            "--gradient-accumulation-steps", str(cfg["grad_accum"]),
        ])
        trained_configs.append({**cfg, "output_dir": output_dir})
else:
    for cfg in TRAIN_CONFIGS:
        if cfg["name"] not in ACTIVE_TRAIN_CONFIG_NAMES:
            continue
        output_dir = CHECKPOINTS_DIR / cfg["name"]
        if output_dir.exists():
            trained_configs.append({**cfg, "output_dir": output_dir})

print("Adapters available for eval:")
for cfg in trained_configs:
    print(cfg["name"], "->", cfg["output_dir"])


## 10. Evaluate Trained Adapters

This uses the exact same evaluator as zero-shot, so the comparison is apples-to-apples.

In [ ]:
adapter_eval_rows = []

if RUN_ADAPTER_EVAL:
    for cfg in trained_configs:
        name = cfg["name"]
        adapter_path = cfg["output_dir"]
        output_path = RUNS_DIR / f"{name}_outputs.jsonl"
        metrics_path = RUNS_DIR / f"{name}_metrics.json"
        train_metrics_path = adapter_path / "train_metrics.json"

        run_command([
            "python", "-B", "scripts/evaluate_student.py",
            "--model-name", MODEL_NAME,
            "--adapter-path", adapter_path,
            "--input-path", EVAL_INPUT_PATH,
            "--output-path", output_path,
            "--metrics-path", metrics_path,
            "--num-samples", str(FINAL_EVAL_NUM_SAMPLES),
            "--batch-size", str(EVAL_BATCH_SIZE),
            "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
        ])
        adapter_eval_rows.append(metric_row(name, metrics_path, train_metrics_path))

adapter_eval_rows

## 11. Compare Metrics

Prioritize `usable_rate` first, then strict `accuracy`, then `format_valid_rate`. Use `loose_math_accuracy` only as a diagnostic for zero-shot/base-model math ability when the model ignores the required XML format.

In [ ]:
comparison_rows = []
if baseline_metrics.exists():
    comparison_rows.append(metric_row("zero_shot", baseline_metrics))

for cfg in TRAIN_CONFIGS:
    name = cfg["name"]
    metrics_path = RUNS_DIR / f"{name}_metrics.json"
    train_metrics_path = CHECKPOINTS_DIR / name / "train_metrics.json"
    if metrics_path.exists():
        comparison_rows.append(metric_row(name, metrics_path, train_metrics_path))

comparison = pd.DataFrame(comparison_rows)
if not comparison.empty:
    display(
        comparison.sort_values(
            by=["usable_rate", "accuracy", "format_valid_rate"],
            ascending=False,
        )
    )
    comparison.to_csv(RUNS_DIR / "comparison.csv", index=False)
    print(f"Saved comparison to {RUNS_DIR / 'comparison.csv'}")
else:
    print("No metrics found yet.")

## 12. Download Outputs in Colab

Run this at the end of a Colab session to download metrics and sampled outputs.

In [ ]:
# Optional Colab download.
# import shutil
# archive = shutil.make_archive("student_sft_results", "zip", RUNS_DIR)
# from google.colab import files
# files.download(archive)